In [1]:
using MarineHydro
using PyCall
cpt = pyimport("capytaine")

# Description of problem
h = Inf # sea depth [m]
omegas = [1.03, 1.7] # frequencies [rad/s]
beta = 0 # incident wave angle [rad]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)


# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
cptbody.center_of_mass = (0.0, 0.0, 0.0)
cptbody.rotation_center = (1.0, 1.0, 0.0) # off set for nonzero off-diagoinal elements
foreach(dof -> cptbody.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody.add_rotation_dof(name=dof), r_DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Horizontal Cylinder"


# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptbody.active_dofs
xr = pyimport("xarray")
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => [0.0], "radiating_dof" => DOFs))
results = cpt.BEMSolver().fill_dataset(test_matrix, cptbody, method="direct")
 

# Get Capytaine values
A_cpt = results.added_mass
B_cpt = results.radiation_damping
F_FK_cpt = results.Froude_Krylov_force 
F_D_cpt = results.diffraction_force


Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.DataArray 'diffraction_force' (omega: 2, wave_direction: 1,
                                       influenced_dof: 2)> Size: 64B
array([[[ -6743.84717853-1903.32445886j,  -6766.41387213+1473.77719906j]],

       [[-11852.5317173 -6795.65856077j, -12808.28557251+2534.67162654j]]])
Coordinates:
  * omega           (omega) float64 16B 1.03 1.7
  * wave_direction  (wave_direction) float64 8B 0.0
  * influenced_dof  (influenced_dof) category 18B Heave Pitch
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
    freq            (omega) float64 16B 0.1639 0.2706
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Diffraction force

In [2]:
A_cpt

PyObject <xarray.DataArray 'added_mass' (omega: 2, radiating_dof: 2, influenced_dof: 2)> Size: 64B
array([[[6818.23611669, 6818.23611669],
        [6818.23611669, 9951.66185187]],

       [[5305.90764352, 5305.90764352],
        [5305.90764352, 8670.68965222]]])
Coordinates:
  * omega           (omega) float64 16B 1.03 1.7
  * radiating_dof   (radiating_dof) category 18B Heave Pitch
  * influenced_dof  (influenced_dof) category 18B Heave Pitch
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
    freq            (omega) float64 16B 0.1639 0.2706
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Added mass

In [3]:
# Get MarineHydro values
mesh = Mesh(cptmesh)
rigid_dof_list = DOFs
rotation_center = collect(cptbody.rotation_center)
fb = FloatingBody(mesh, rigid_dof_list, rotation_center)

A_and_B = [calculate_radiation_forces(fb, omega) for omega in omegas]
A_zip, B_zip = zip(A_and_B...)
A_mh = merge(A_zip...)
B_mh = merge(B_zip...)
F_FK_mh = merge([FroudeKrylovForce(fb, omega) for omega in omegas]...)
F_D_mh = merge([DiffractionForce(fb,omega) for omega in omegas]...)


Dict{Tuple{Any, String}, Any} with 4 entries:
  (1.03, "Heave") => -6746.09-1904.15im
  (1.7, "Pitch")  => -12833.2+2537.93im
  (1.03, "Pitch") => -6768.71+1471.82im
  (1.7, "Heave")  => -11874.4-6786.85im

In [4]:
A_mh

Dict{Tuple{Any, String, String}, Any} with 8 entries:
  (1.03, "Pitch", "Heave") => 6978.06
  (1.7, "Heave", "Heave")  => 5435.58
  (1.7, "Heave", "Pitch")  => 5435.58
  (1.03, "Heave", "Heave") => 6978.06
  (1.03, "Heave", "Pitch") => 6978.06
  (1.7, "Pitch", "Pitch")  => 8876.02
  (1.03, "Pitch", "Pitch") => 10182.8
  (1.7, "Pitch", "Heave")  => 5435.58

In [5]:
A_cpt

PyObject <xarray.DataArray 'added_mass' (omega: 2, radiating_dof: 2, influenced_dof: 2)> Size: 64B
array([[[6818.23611669, 6818.23611669],
        [6818.23611669, 9951.66185187]],

       [[5305.90764352, 5305.90764352],
        [5305.90764352, 8670.68965222]]])
Coordinates:
  * omega           (omega) float64 16B 1.03 1.7
  * radiating_dof   (radiating_dof) category 18B Heave Pitch
  * influenced_dof  (influenced_dof) category 18B Heave Pitch
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
    freq            (omega) float64 16B 0.1639 0.2706
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Added mass

In [6]:
A_cpt.sel(omega=1.03, 
    radiating_dof="Pitch", 
    influenced_dof="Heave").values[]

6818.236116694181

In [7]:
A_mh[(1.03, "Heave", "Pitch")]

6978.062210908978

In [8]:
using Test

# Round very small values to zero
clean(x) = abs(x) < 1e-2 ? 0.0 : x

@testset "Comparison of MDOF Horizontal Cylinder Forces with Capytaine (rtol = 1e-1) " begin
    for omega in omegas
        for influenced_dof in DOFs

            for radiating_dof in DOFs
                @testset "Omega: $omega, influenced_dof: $influenced_dof, radiating_dof: $radiating_dof" begin
                    # Test added mass
                    a_cpt = A_cpt.sel(omega=omega, radiating_dof=radiating_dof, influenced_dof=influenced_dof).values[]
                    a_mh = A_mh[(omega, influenced_dof, radiating_dof)]
                    @test  clean(a_cpt) ≈ clean(a_mh) rtol=1e-1  
                    # Test radiation damping
                    b_cpt = B_cpt.sel(omega=omega, radiating_dof=radiating_dof, influenced_dof=influenced_dof).values[]
                    b_mh = B_mh[(omega, influenced_dof, radiating_dof)]
                    @test  clean(b_cpt) ≈ clean(b_mh) rtol=1e-1
                end                          
            end
            @testset "Omega: $omega, influenced_dof: $influenced_dof" begin
                # Test FK force
                f_FK_cpt = F_FK_cpt.sel(omega=omega, influenced_dof=influenced_dof).values[]
                f_FK_mh = F_FK_mh[(omega, influenced_dof)]
                @test clean(real(f_FK_cpt)) ≈ clean(real(f_FK_mh)) rtol=1e-1
                @test clean(imag(f_FK_cpt)) ≈ clean(imag(f_FK_mh)) rtol=1e-1
                # Test diffraction force
                f_D_cpt = F_D_cpt.sel(omega=omega, influenced_dof=influenced_dof).values[]
                f_D_mh = F_D_mh[(omega, influenced_dof)]
                @test clean(real(f_D_cpt)) ≈ clean(real(f_D_mh)) rtol=1e-1
                @test clean(imag(f_D_cpt)) ≈ clean(imag(f_D_mh)) rtol=1e-1
            end            
        end        
    end
end


Test Summary:                                                               | Pass  Total  Time
Comparison of MDOF Horizontal Cylinder Forces with Capytaine (rtol = 1e-1)  |   32     32  0.3s


Test.DefaultTestSet("Comparison of MDOF Horizontal Cylinder Forces with Capytaine (rtol = 1e-1) ", Any[Test.DefaultTestSet("Omega: 1.03, influenced_dof: Heave, radiating_dof: Heave", Any[], 2, false, false, true, 1.772117921498e9, 1.772117921633e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl"), Test.DefaultTestSet("Omega: 1.03, influenced_dof: Heave, radiating_dof: Pitch", Any[], 2, false, false, true, 1.772117921633e9, 1.772117921635e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl"), Test.DefaultTestSet("Omega: 1.03, influenced_dof: Heave", Any[], 4, false, false, true, 1.772117921635e9, 1.772117921766e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl"), Test.DefaultTestSet("Omega: 1.03, influenced_dof: Pitch, radiating_dof: Heave", Any[], 2, false, false, true, 1.772117921766e9, 1.772

In [9]:
# Now test derivatives 
using Zygote
A(w) = calculate_radiation_forces(fb, w)[1]

A (generic function with 1 method)

In [10]:
A(1.04)

Dict{Tuple{Any, String, String}, Any} with 4 entries:
  (1.04, "Pitch", "Heave") => 6958.5
  (1.04, "Heave", "Heave") => 6958.5
  (1.04, "Heave", "Pitch") => 6958.5
  (1.04, "Pitch", "Pitch") => 10165.5

In [26]:
num_DOFs = length(DOFs)

function A_vec(w)
    dict = calculate_radiation_forces(fb, w)[1]
    matrix = [real.(dict[(w, i, r)]) for i in DOFs, r in DOFs]
    vector = vec(matrix)
    return vector
end

A_vec (generic function with 1 method)

In [27]:
A_vec(1.06)

4-element Vector{Float64}:
  6918.54668616958
  6918.54668616958
  6918.546686169581
 10130.226097442142

In [ ]:

j = Zygote.jacobian(1.04 + 0im) do w
    # Use real(w) for the physics code, but Zygote still tracks the complex sensitivity
    A_vec(real(w)) 
end

(ComplexF64[-1970.5168032636016 + 0.0im, -1970.516803263605 + 0.0im, -1970.5168032635968 + 0.0im, -1741.325123502521 + 0.0im],)

In [25]:
A_w_grad = Zygote.gradient(w -> A_vec(w)[4],1.04)

(-1741.325123502521,)

In [32]:
function A_and_B_vec(w)
    added_mass_dict, damping_dict = calculate_radiation_forces(fb, w)
    A_vals = [real(added_mass_dict[(w, i, r)]) for i in DOFs, r in DOFs]
    B_vals = [real(damping_dict[(w, i, r)]) for i in DOFs, r in DOFs]
    return vcat(vec(A_vals), vec(B_vals)) # Note this is [A_11,A_12,A_21...,B_22]
end

A_and_B_vec (generic function with 1 method)

In [34]:
A_and_B_vec(1.04)

8-element Vector{Float64}:
  6958.495778472966
  6958.495778472966
  6958.495778472965
 10165.512479367835
  1903.537895292977
  1903.5378952929773
  1903.5378952929768
  1912.6311678631964

In [35]:
j = Zygote.jacobian(1.04 + 0im) do w
    # Use real(w) for the physics code, but Zygote still tracks the complex sensitivity
    A_and_B_vec(real(w)) 
end

(ComplexF64[-1970.5168032636016 + 0.0im, -1970.516803263605 + 0.0im, -1970.5168032635968 + 0.0im, -1741.325123502521 + 0.0im, 3665.7239412671133 + 0.0im, 3665.7239412671197 + 0.0im, 3665.723941267111 + 0.0im, 3727.0318089107072 + 0.0im],)